In [1]:
import os
import pandas as pd
utkface_path = r'c:\Users\Acer\OneDrive\Desktop\DeepLearning\UTKFace'

ages = []
genders = []
images = []

for filename in os.listdir(utkface_path):
    if filename.endswith('.jpg'):
        try:
            parts = filename.split('_')

            age = int(parts[0])

            gender = int(parts[1])

            ages.append(age)
            genders.append(gender)
            images.append(filename)
        except (ValueError, IndexError):
            continue

df = pd.DataFrame({
    'age': ages,
    'gender': genders,
    'img': images
})


In [2]:
df['age'] = df['age'] / 100.0

print(f"Age range after normalization: {df['age'].min():.2f} to {df['age'].max():.2f}")
print(f"Sample ages: {df['age'].head().values}")

Age range after normalization: 0.01 to 1.16
Sample ages: [1. 1. 1. 1. 1.]


In [3]:
df.head()

,age,gender,img
0,1.0,0,100_0_0_20170112213500903.jpg.chip.jpg
1,1.0,0,100_0_0_20170112215240346.jpg.chip.jpg
2,1.0,1,100_1_0_20170110183726390.jpg.chip.jpg
3,1.0,1,100_1_0_20170112213001988.jpg.chip.jpg
4,1.0,1,100_1_0_20170112213303693.jpg.chip.jpg


In [4]:
len(ages)

20333

In [5]:
df.shape

(20333, 3)

In [6]:
train_df = df.iloc[:17000 , :]
test_df = df.iloc[17000:, :]

In [7]:
train_df.shape

(17000, 3)

In [8]:
test_df.shape

(3333, 3)

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# No data augmentation - using real images only
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

c:\Users\Acer\anaconda3\envs\py310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [10]:
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    directory=utkface_path,
    x_col='img',
    y_col=['age', 'gender'],  # dictionary instead of list
    target_size=(200, 200),
    class_mode='multi_output'
)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    directory=utkface_path,
    x_col='img',
    y_col=['age', 'gender'],
    target_size=(200, 200),
    class_mode='multi_output'
)

Found 17000 validated image filenames.
Found 3333 validated image filenames.


In [11]:
# Simple function to split age and gender
import numpy as np


def multi_output_generator(generator):
    for batch_x, batch_y in generator:
        # batch_y is already a list: [ages_array, genders_array]
        # make sure each is a numpy array
        ages = np.array(batch_y[0])
        genders = np.array(batch_y[1])
        yield batch_x, {'age': ages, 'gender': genders}

# Apply the wrapper
train_gen_wrapped = multi_output_generator(train_generator)
test_gen_wrapped = multi_output_generator(test_generator)

In [12]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

In [13]:
resnet = ResNet50(include_top= False , input_shape= (200, 200 , 3))

In [18]:
from tensorflow.keras.layers import GlobalAveragePooling2D
resnet.trainable = False 
output= resnet.layers[-1].output

flatten = Flatten()(output)

dense1= Dense(256 , activation= 'relu')(flatten)
dense2= Dense(256 , activation= 'relu')(flatten)

dense3= Dense(256 , activation= 'relu')(dense1)
dense4= Dense(256 , activation= 'relu')(dense2)

output1= Dense(1, activation= 'linear', name= 'age')(dense3)
output2= Dense(1,activation= 'sigmoid', name= 'gender')(dense4)

In [19]:
model = Model(inputs= resnet.input , outputs= [output1 , output2])

In [20]:
model.compile(optimizer= 'adam' , loss= {'age': 'mae' , 'gender': 'binary_crossentropy'}, metrics= {'age': 'mae' , 'gender': 'accuracy'},loss_weights={'age':1,'gender':1})

In [21]:
model.fit(
    train_gen_wrapped,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=test_gen_wrapped,
    validation_steps=len(test_generator)
)

Epoch 1/10
532/532 [==============================] - 49s 85ms/step - loss: 1.6202 - age_loss: 0.6883 - gender_loss: 0.9319 - age_mae: 0.6883 - gender_accuracy: 0.5649 - val_loss: 0.8220 - val_age_loss: 0.1892 - val_gender_loss: 0.6327 - val_age_mae: 0.1892 - val_gender_accuracy: 0.6751
Epoch 2/10
532/532 [==============================] - 44s 83ms/step - loss: 0.7666 - age_loss: 0.0910 - gender_loss: 0.6756 - age_mae: 0.0910 - gender_accuracy: 0.5621 - val_loss: 0.9639 - val_age_loss: 0.2526 - val_gender_loss: 0.7113 - val_age_mae: 0.2526 - val_gender_accuracy: 0.3342
Epoch 3/10
532/532 [==============================] - 44s 83ms/step - loss: 0.7754 - age_loss: 0.0864 - gender_loss: 0.6890 - age_mae: 0.0864 - gender_accuracy: 0.5170 - val_loss: 0.9575 - val_age_loss: 0.2593 - val_gender_loss: 0.6982 - val_age_mae: 0.2593 - val_gender_accuracy: 0.3330
Epoch 4/10
532/532 [==============================] - 44s 83ms/step - loss: 0.7785 - age_loss: 0.0855 - gender_loss: 0.6929 - age_mae: 0

In [27]:
from typing import List 
def removeElement(nums: List[int], val: int) -> int:
      
      output = []
      for num in nums:
            if num == val:
                continue
            else:
                output.append(num)
      return  output


input= [3, 2, 2, 3]
val= 3

print(removeElement(input , val))

[2, 2]
